In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.remote("sc://localhost:15005").getOrCreate()
spark.sql("SELECT 'Connected to Spark!' as status").show()

+-------------------+
|             status|
+-------------------+
|Connected to Spark!|
+-------------------+



In [2]:
spark.sql("SET spark.sql.catalogImplementation").show(truncate=False)

+-------------------------------+-----+
|key                            |value|
+-------------------------------+-----+
|spark.sql.catalogImplementation|hive |
+-------------------------------+-----+



In [3]:
raw_df = spark.read\
    .option("header", True)\
    .option("inferSchema", False)\
    .csv("hdfs:///data/sales/amazon_products_sales_data_uncleaned.csv")

print("Total rows loaded:", raw_df.count())
print("Total columns:", len(raw_df.columns))
raw_df.printSchema()

Total rows loaded: 42675
Total columns: 16
root
 |-- title: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- number_of_reviews: string (nullable = true)
 |-- bought_in_last_month: string (nullable = true)
 |-- current/discounted_price: string (nullable = true)
 |-- price_on_variant: string (nullable = true)
 |-- listed_price: string (nullable = true)
 |-- is_best_seller: string (nullable = true)
 |-- is_sponsored: string (nullable = true)
 |-- is_couponed: string (nullable = true)
 |-- buy_box_availability: string (nullable = true)
 |-- delivery_details: string (nullable = true)
 |-- sustainability_badges: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- product_url: string (nullable = true)
 |-- collected_at: string (nullable = true)



In [4]:
raw_df.show(5, truncate=80)

+--------------------------------------------------------------------------------+------------------+-----------------+-------------------------+------------------------+----------------------------+------------+--------------+------------+---------------------+--------------------+--------------------+---------------------+--------------------------------------------------------------+--------------------------------------------------------------------------------+-------------------+
|                                                                           title|            rating|number_of_reviews|     bought_in_last_month|current/discounted_price|            price_on_variant|listed_price|is_best_seller|is_sponsored|          is_couponed|buy_box_availability|    delivery_details|sustainability_badges|                                                     image_url|                                                                     product_url|       collected_at|
+-----------------

In [5]:
print("Null counts per column:")
raw_df.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)).alias(c)
    for c in raw_df.columns
]).show(vertical=True)

Null counts per column:
-RECORD 0-------------------------
 title                    | 0     
 rating                   | 885   
 number_of_reviews        | 885   
 bought_in_last_month     | 3068  
 current/discounted_price | 11386 
 price_on_variant         | 34    
 listed_price             | 107   
 is_best_seller           | 112   
 is_sponsored             | 14    
 is_couponed              | 199   
 buy_box_availability     | 13726 
 delivery_details         | 11388 
 sustainability_badges    | 36259 
 image_url                | 660   
 product_url              | 3298  
 collected_at             | 968   



In [6]:
total = raw_df.count()
distinct = raw_df.dropDuplicates().count()
duplicates = total - distinct

print(f"Total rows     : {total}")
print(f"Distinct rows  : {distinct}")
print(f"Duplicate rows : {duplicates}")

Total rows     : 42675
Distinct rows  : 40033
Duplicate rows : 2642


In [7]:
print("=== rating samples ===")
raw_df.select("rating").distinct().show(10, truncate=False)

print("=== number_of_reviews samples ===")
raw_df.select("number_of_reviews").distinct().show(10, truncate=False)

print("=== bought_in_last_month samples ===")
raw_df.select("bought_in_last_month").distinct().show(10, truncate=False)

print("=== current/discounted_price samples ===")
raw_df.select("current/discounted_price").distinct().show(10, truncate=False)

print("=== listed_price samples ===")
raw_df.select("listed_price").distinct().show(10, truncate=False)

=== rating samples ===
+----------------------------------------+
|rating                                  |
+----------------------------------------+
| Live Streaming                         |
| 2-Way Boat & Car Audio Speaker         |
| Pack Of 100"                           |
| Up to 20 LBS                           |
| AI Tracking                            |
| Digital Controls + 10 Heat Settings    |
| 11th Gen Intel Core i3-1115G4 Processor|
| DCI-P3 90%                             |
| 16' L"                                 |
| Auto 2-Sided Printing - Black"         |
+----------------------------------------+
only showing top 10 rows

=== number_of_reviews samples ===
+-----------------+
|number_of_reviews|
+-----------------+
|28,351           |
|12,564           |
|2,782            |
|36,643           |
|2,703            |
|691              |
|8,104            |
|1,609            |
|829              |
|15,448           |
+-----------------+
only showing top 10 rows

=== bough

In [8]:
df = raw_df.drop("image_url", "product_url", "price_on_variant", "delivery_details")

print("Columns after drop:", df.columns)
print("Column count:", len(df.columns))

Columns after drop: ['title', 'rating', 'number_of_reviews', 'bought_in_last_month', 'current/discounted_price', 'listed_price', 'is_best_seller', 'is_sponsored', 'is_couponed', 'buy_box_availability', 'sustainability_badges', 'collected_at']
Column count: 12


In [9]:
df = df.withColumn(
    "rating",
    F.when(
        F.col("rating").rlike(r"^\d+\.?\d*\s+out of 5"),
        F.regexp_extract(F.col("rating"), r"^(\d+\.?\d*)", 1).cast("float")
    ).otherwise(F.lit(None))
)

print("Rating after cleaning:")
df.select("rating").summary("count", "min", "max", "mean").show()

Rating after cleaning:
+-------+-----------------+
|summary|           rating|
+-------+-----------------+
|  count|            38487|
|    min|              1.0|
|    max|              5.0|
|   mean|4.396663805333731|
+-------+-----------------+



In [10]:
print("Ratings above 5 (should be 0):")
df.filter(F.col("rating") > 5).select("rating").show()

print("Null ratings:", df.filter(F.col("rating").isNull()).count())

print("Rating distribution (valid values only):")
df.filter(F.col("rating").isNotNull())\
  .groupBy("rating").count().orderBy("rating").show(20)

Ratings above 5 (should be 0):
+------+
|rating|
+------+
+------+

Null ratings: 4188
Rating distribution (valid values only):
+------+-----+
|rating|count|
+------+-----+
|   1.0|    4|
|   1.5|   15|
|   2.0|  143|
|   2.3|    1|
|   2.4|    3|
|   2.5|    1|
|   2.6|    1|
|   2.7|   15|
|   2.8|    4|
|   2.9|    2|
|   3.0|  148|
|   3.1|    2|
|   3.2|  363|
|   3.3|    6|
|   3.4|  216|
|   3.5|  156|
|   3.6|  663|
|   3.7|  454|
|   3.8|  924|
|   3.9| 1301|
+------+-----+
only showing top 20 rows



In [11]:
df = df.withColumn(
    "number_of_reviews",
    F.regexp_replace(F.col("number_of_reviews"), ",", "").cast("integer")
)

print("Reviews after cleaning:")
df.select("number_of_reviews").summary("count", "min", "max", "mean").show()

Reviews after cleaning:
+-------+------------------+
|summary| number_of_reviews|
+-------+------------------+
|  count|             38583|
|    min|                 1|
|    max|            865598|
|   mean|3212.3269315501648|
+-------+------------------+



In [13]:
print("Null reviews count:", df.filter(F.col("number_of_reviews").isNull()).count())

print("Top 10 most reviewed products:")
df.select("title", "number_of_reviews")\
  .orderBy(F.col("number_of_reviews").desc())\
  .show(10, truncate=60)

Null reviews count: 4092
Top 10 most reviewed products:
+------------------------------------------------------------+-----------------+
|                                                       title|number_of_reviews|
+------------------------------------------------------------+-----------------+
|Amazon Basics 48-Pack AA Alkaline High-Performance Batter...|           865598|
|SanDisk 32GB Ultra microSDHC UHS-I Memory Card with Adapt...|           645418|
|[Older Version] SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I...|           645416|
|Amazon Basics AAA Alkaline High-Performance Batteries, 1....|           625776|
|Amazon Basics HDMI Cable, 3ft, 4K@60Hz, High-Speed 4K HDM...|           553927|
|Amazon Fire TV Stick, sharp picture quality, fast streami...|           517617|
|SanDisk 1TB Extreme microSDXC UHS-I Memory Card with Adap...|           353306|
|[Older Version] SanDisk 128GB Ultra microSDXC UHS-I Memor...|           315834|
|Blink Mini - Compact indoor plug-in smart security c

In [14]:
df = df.withColumn(
    "bought_in_last_month",
    F.regexp_extract(F.col("bought_in_last_month"), r"(\d+)", 1).cast("integer")
)

print("bought_in_last_month after cleaning:")
df.select("bought_in_last_month").summary("count", "min", "max", "mean").show()

bought_in_last_month after cleaning:
+-------+--------------------+
|summary|bought_in_last_month|
+-------+--------------------+
|  count|               32124|
|    min|                   0|
|    max|               28371|
|   mean|  159.86281285020544|
+-------+--------------------+



In [16]:
print("Suspicious prices (above 10 000$):")
df.filter(F.col("current_price") > 10000)\
  .select("title", "current_price")\
  .orderBy(F.col("current_price").desc())\
  .show(10, truncate=60)

print("Zero or negative prices:")
df.filter(F.col("current_price") <= 0)\
  .select("title", "current_price")\
  .show(10, truncate=60)

Suspicious prices (above 10 000$):
+------------------------------------------------------------+-------------+
|                                                       title|current_price|
+------------------------------------------------------------+-------------+
|Mounting Dream UL Listed Advanced Tilt TV Wall Mount for ...| 6.0040009E12|
|Pendaflex Extra Capacity Reinforced Hanging File Folders, 2"|  2.5041522E7|
|Pendaflex Extra Capacity Reinforced Hanging File Folders, 2"|  2.5041522E7|
|"Lenovo ThinkPad P14s Gen 5 AMD AMD Ryzen™ 7 PRO 8840HS P...|    3251164.0|
|                            Dell Latitude 3190 2-in-1,11.6 "|    1366768.0|
|               "Mounting Dream TV Wall Mount for 42-86"" TVs|     800400.0|
|               "Mounting Dream TV Wall Mount for 42-86"" TVs|     800400.0|
|"Lenovo ThinkPad E14 Gen 6 Business Laptop (14"" FHD+ Ant...|     671355.0|
|                         "HP 14"" HD Student Business Laptop|     128256.0|
|"SAMSUNG Galaxy Tab A9 (64GB, 4GB, Wi-Fi

In [17]:
df = df.withColumn(
    "listed_price",
    F.regexp_replace(F.col("listed_price"), r"[^\d.]", "").cast("float")
)

print("listed_price after cleaning:")
df.select("listed_price").summary("count", "min", "max", "mean").show()

listed_price after cleaning:
+-------+------------------+
|summary|      listed_price|
+-------+------------------+
|  count|             12692|
|    min|               1.0|
|    max|       3.4652024E7|
|   mean|3448.4408958707927|
+-------+------------------+



In [18]:
df = df.withColumn(
    "discount_pct",
    F.round(
        (F.col("listed_price") - F.col("current_price")) / F.col("listed_price") * 100,
        2
    )
)

print("discount_pct stats:")
df.select("discount_pct").summary("count", "min", "max", "mean").show()

discount_pct stats:
+-------+--------------------+
|summary|        discount_pct|
+-------+--------------------+
|  count|               12489|
|    min| -2.4496127899731E11|
|    max|               100.0|
|   mean|-1.96248607918011...|
+-------+--------------------+



In [19]:
print("Negative discounts (current_price > listed_price):")
df.filter(F.col("discount_pct") < -100)\
  .select("title", "current_price", "listed_price", "discount_pct")\
  .orderBy(F.col("discount_pct"))\
  .show(10, truncate=60)

print("Extreme discounts (above 100%):")
df.filter(F.col("discount_pct") > 100)\
  .select("title", "current_price", "listed_price", "discount_pct")\
  .show(10, truncate=60)

Negative discounts (current_price > listed_price):
+------------------------------------------------------------+-------------+------------+-------------------+
|                                                       title|current_price|listed_price|       discount_pct|
+------------------------------------------------------------+-------------+------------+-------------------+
|Mounting Dream UL Listed Advanced Tilt TV Wall Mount for ...| 6.0040009E12|      2451.0|-2.4496127899731E11|
|"Lenovo ThinkPad P14s Gen 5 AMD AMD Ryzen™ 7 PRO 8840HS P...|    3251164.0|        4.25|     -7.649787647E7|
|                            Dell Latitude 3190 2-in-1,11.6 "|    1366768.0|        4.05|     -3.374725767E7|
|"SAMSUNG Galaxy Tab A9 (64GB, 4GB, Wi-Fi Only) 8.7"" Andr...|     110256.0|        4.25|        -2594158.82|
|Pendaflex Extra Capacity Reinforced Hanging File Folders, 2"|  2.5041522E7|      3130.0|         -799948.63|
|Pendaflex Extra Capacity Reinforced Hanging File Folders, 2"|  2.504

In [20]:
df = df.withColumn(
    "collected_at",
    F.to_date(F.col("collected_at"))
)

print("Date range:")
df.select(
    F.min("collected_at").alias("earliest"),
    F.max("collected_at").alias("latest")
).show()

print("Null dates:", df.filter(F.col("collected_at").isNull()).count())

Date range:
+----------+----------+
|  earliest|    latest|
+----------+----------+
|2025-08-21|2025-08-30|
+----------+----------+

Null dates: 3303


In [21]:
before = df.count()

df = df.dropna(subset=["title", "current_price", "listed_price"])

after = df.count()
print(f"Rows before dropna : {before}")
print(f"Rows after dropna  : {after}")
print(f"Rows removed       : {before - after}")

Rows before dropna : 42675
Rows after dropna  : 12489
Rows removed       : 30186


In [22]:
before = df.count()

df = df.dropDuplicates()

after = df.count()
print(f"Rows before dedup : {before}")
print(f"Rows after dedup  : {after}")
print(f"Duplicates removed: {before - after}")

Rows before dedup : 12489
Rows after dedup  : 3153
Duplicates removed: 9336


In [23]:
print("=" * 50)
print("DATA QUALITY REPORT - Before Final Filters")
print("=" * 50)

print(f"\nTotal rows: {df.count()}")

print("\nNull counts per column:")
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(vertical=True)

print("\nKey stats:")
df.select("rating", "current_price", "listed_price", "discount_pct")\
  .summary("count", "min", "max", "mean", "stddev")\
  .show()

DATA QUALITY REPORT - Before Final Filters

Total rows: 3153

Null counts per column:
-RECORD 0---------------------
 title                 | 0    
 rating                | 282  
 number_of_reviews     | 276  
 bought_in_last_month  | 164  
 listed_price          | 0    
 is_best_seller        | 14   
 is_sponsored          | 5    
 is_couponed           | 9    
 buy_box_availability  | 313  
 sustainability_badges | 2609 
 collected_at          | 269  
 current_price         | 0    
 discount_pct          | 0    


Key stats:
+-------+-------------------+--------------------+------------------+--------------------+
|summary|             rating|       current_price|      listed_price|        discount_pct|
+-------+-------------------+--------------------+------------------+--------------------+
|  count|               2871|                3153|              3153|                3153|
|    min|                3.0|                 1.0|               1.0| -2.4496127899731E11|
|    max|   

In [24]:
before = df.count()

df = df.filter(F.col("rating").between(0, 5))
after_rating = df.count()

df = df.filter(F.col("current_price").between(0.01, 10000))
after_price = df.count()

df = df.filter(F.col("listed_price").between(0.01, 10000))
after_listed = df.count()

df = df.filter(F.col("discount_pct").between(-100, 100))
after_discount = df.count()

print(f"Rows before filters        : {before}")
print(f"After rating filter        : {after_rating}  (removed {before - after_rating})")
print(f"After current_price filter : {after_price}  (removed {after_rating - after_price})")
print(f"After listed_price filter  : {after_listed}  (removed {after_price - after_listed})")
print(f"After discount_pct filter  : {after_discount}  (removed {after_listed - after_discount})")
print(f"\nFinal clean row count      : {after_discount}")

Rows before filters        : 3153
After rating filter        : 2871  (removed 282)
After current_price filter : 2871  (removed 0)
After listed_price filter  : 2871  (removed 0)
After discount_pct filter  : 2871  (removed 0)

Final clean row count      : 2871


In [25]:
print("=" * 50)
print("DATA QUALITY REPORT - After Final Filters")
print("=" * 50)

df.select("rating", "current_price", "listed_price", "discount_pct")\
  .summary("count", "min", "max", "mean")\
  .show()

print("Sample of clean data:")
df.show(5, truncate=60)

DATA QUALITY REPORT - After Final Filters
+-------+-----------------+------------------+------------------+------------------+
|summary|           rating|     current_price|      listed_price|      discount_pct|
+-------+-----------------+------------------+------------------+------------------+
|  count|             2871|              2871|              2871|              2871|
|    min|              3.0|              2.97|              4.95|               4.5|
|    max|              5.0|            4399.0|            5399.0|             85.42|
|   mean|4.455938683117528|136.60663857579604|167.90901760156459|21.062988505747104|
+-------+-----------------+------------------+------------------+------------------+

Sample of clean data:
+------------------------------------------------------------+------+-----------------+--------------------+------------+--------------+------------+-----------+--------------------+---------------------+------------+-------------+------------+
|         

In [26]:
df = df.withColumn(
    "price_tier",
    F.when(F.col("current_price") < 30, "Budget")
     .when(F.col("current_price") < 100, "Mid-Range")
     .when(F.col("current_price") < 500, "Premium")
     .otherwise("Luxury")
)

print("Distribution by price tier:")
df.groupBy("price_tier")\
  .agg(
      F.count("*").alias("count"),
      F.round(F.avg("current_price"), 2).alias("avg_price"),
      F.round(F.avg("rating"), 2).alias("avg_rating"),
      F.round(F.avg("discount_pct"), 2).alias("avg_discount_pct")
  )\
  .orderBy("avg_price")\
  .show()

Distribution by price tier:
+----------+-----+---------+----------+----------------+
|price_tier|count|avg_price|avg_rating|avg_discount_pct|
+----------+-----+---------+----------+----------------+
|    Budget| 1049|    17.73|      4.54|           24.52|
| Mid-Range|  837|    59.41|      4.43|           20.77|
|   Premium|  863|   212.18|      4.38|           18.17|
|    Luxury|  122|  1153.75|      4.41|           13.83|
+----------+-----+---------+----------+----------------+



In [27]:
dim_products = df.select(
    "title",
    "is_best_seller",
    "is_sponsored",
    "is_couponed",
    "sustainability_badges",
    "buy_box_availability"
).distinct()\
 .withColumn("product_id", F.monotonically_increasing_id())

print("dim_products row count:", dim_products.count())
dim_products.show(5, truncate=60)

dim_products row count: 2722
+------------------------------------------------------------+--------------+------------+-----------+-----------------------+--------------------+----------+
|                                                       title|is_best_seller|is_sponsored|is_couponed|  sustainability_badges|buy_box_availability|product_id|
+------------------------------------------------------------+--------------+------------+-----------+-----------------------+--------------------+----------+
|TP-Link Tapo 1080P Outdoor Wired Pan/Tilt Security Wi-Fi ...|      No Badge|     Organic|  No Coupon|       Works with Alexa|         Add to cart|         0|
|Kasa Smart Indoor Pan-Tilt Home Security Camera, 1080p HD...|   Best Seller|     Organic|  No Coupon|       Works with Alexa|         Add to cart|         1|
|Tapo TP-Link 2K Pan/Tilt Indoor Security Camera for Baby ...|      No Badge|     Organic|  No Coupon|       Works with Alexa|         Add to cart|         2|
|TP-Link AX1800 W

In [28]:
dim_time = df.select("collected_at").distinct()\
    .withColumn("date_id", F.monotonically_increasing_id())\
    .withColumn("year",    F.year("collected_at"))\
    .withColumn("month",   F.month("collected_at"))\
    .withColumn("day",     F.dayofmonth("collected_at"))

print("dim_time row count:", dim_time.count())
dim_time.show()

dim_time row count: 6
+------------+-------+----+-----+---+
|collected_at|date_id|year|month|day|
+------------+-------+----+-----+---+
|  2025-08-25|      0|2025|    8| 25|
|  2025-08-21|      1|2025|    8| 21|
|  2025-08-27|      2|2025|    8| 27|
|  2025-08-24|      3|2025|    8| 24|
|  2025-08-30|      4|2025|    8| 30|
|  2025-08-29|      5|2025|    8| 29|
+------------+-------+----+-----+---+



In [29]:
dim_price_tier = df.select("price_tier").distinct()\
    .withColumn("tier_id", F.monotonically_increasing_id())

print("dim_price_tier row count:", dim_price_tier.count())
dim_price_tier.show()

dim_price_tier row count: 4
+----------+-------+
|price_tier|tier_id|
+----------+-------+
|   Premium|      0|
|    Budget|      1|
|    Luxury|      2|
| Mid-Range|      3|
+----------+-------+



In [30]:
df = df.join(dim_products.select("title", "product_id"), on="title", how="left")
df = df.join(dim_time.select("collected_at", "date_id"), on="collected_at", how="left")
df = df.join(dim_price_tier.select("price_tier", "tier_id"), on="price_tier", how="left")

print("Null product_id:", df.filter(F.col("product_id").isNull()).count())
print("Null date_id:", df.filter(F.col("date_id").isNull()).count())
print("Null tier_id:", df.filter(F.col("tier_id").isNull()).count())

Null product_id: 0
Null date_id: 0
Null tier_id: 0


In [31]:
fact_sales = df.select(
    F.monotonically_increasing_id().alias("sale_id"),
    "product_id",
    "date_id",
    "tier_id",
    "current_price",
    "listed_price",
    "discount_pct",
    "rating",
    "number_of_reviews",
    "bought_in_last_month"
)

print("fact_sales row count:", fact_sales.count())
fact_sales.printSchema()
fact_sales.show(5)

fact_sales row count: 3291
root
 |-- sale_id: long (nullable = false)
 |-- product_id: long (nullable = true)
 |-- date_id: long (nullable = true)
 |-- tier_id: long (nullable = true)
 |-- current_price: float (nullable = true)
 |-- listed_price: float (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- rating: float (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- bought_in_last_month: integer (nullable = true)

+-------+----------+-------+-------+-------------+------------+------------+------+-----------------+--------------------+
|sale_id|product_id|date_id|tier_id|current_price|listed_price|discount_pct|rating|number_of_reviews|bought_in_last_month|
+-------+----------+-------+-------+-------------+------------+------------+------+-----------------+--------------------+
|      0|       244|      4|      0|       299.99|      429.99|       30.23|   3.9|             2154|                 300|
|      1|        70|      4|      1|        23.98|  

In [32]:
fact_sales.write.mode("overwrite").saveAsTable("fact_sales")
print("fact_sales saved ✅")

dim_products.write.mode("overwrite").saveAsTable("dim_products")
print("dim_products saved ✅")

dim_time.write.mode("overwrite").saveAsTable("dim_time")
print("dim_time saved ✅")

dim_price_tier.write.mode("overwrite").saveAsTable("dim_price_tier")
print("dim_price_tier saved ✅")

fact_sales saved ✅
dim_products saved ✅
dim_time saved ✅
dim_price_tier saved ✅


In [33]:
print("Tables in Metastore:")
spark.sql("SHOW TABLES").show()

Tables in Metastore:
+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|  default|dim_price_tier|      false|
|  default|  dim_products|      false|
|  default|      dim_time|      false|
|  default|    fact_sales|      false|
+---------+--------------+-----------+



In [34]:
print("=== Top 10 Most Reviewed Products ===")
spark.sql("""
    SELECT p.title, f.rating, f.number_of_reviews, f.current_price
    FROM fact_sales f
    JOIN dim_products p ON f.product_id = p.product_id
    ORDER BY f.number_of_reviews DESC
    LIMIT 10
""").show(truncate=50)

print("=== Revenue by Price Tier ===")
spark.sql("""
    SELECT t.price_tier,
           COUNT(*) as products,
           ROUND(AVG(f.rating), 2) as avg_rating,
           ROUND(AVG(f.discount_pct), 2) as avg_discount,
           ROUND(AVG(f.current_price), 2) as avg_price
    FROM fact_sales f
    JOIN dim_price_tier t ON f.tier_id = t.tier_id
    GROUP BY t.price_tier
    ORDER BY avg_price
""").show()

print("=== Date Breakdown ===")
spark.sql("""
    SELECT t.year, t.month, t.day, COUNT(*) as records
    FROM fact_sales f
    JOIN dim_time t ON f.date_id = t.date_id
    GROUP BY t.year, t.month, t.day
    ORDER BY t.day
""").show()

=== Top 10 Most Reviewed Products ===
+--------------------------------------------------+------+-----------------+-------------+
|                                             title|rating|number_of_reviews|current_price|
+--------------------------------------------------+------+-----------------+-------------+
|ARRIS (S33) - Cable Modem - Fast DOCSIS 3.1 Mul...|   4.7|           645416|        12.46|
|OWC 4TB Envoy Ultra Thunderbolt 5 Portable SSD ...|   4.7|           219712|        19.99|
|TP-Link EAP650 Ultra-Slim Wireless Access Point...|   4.6|           205678|         9.34|
|Pyle 20W Megaphone Bullhorn - 5.4'' x 8.6'' Por...|   4.7|           199598|        57.99|
|                   Insta360 X5 Premium Lens Guards|   4.6|           185776|        14.95|
|Sony BRAVIA Theater Bar 6, 3.1.2ch Sound bar wi...|   4.4|           167901|         9.83|
|Logitech C930e 1080P HD Video Webcam - 90-Degre...|   4.4|           139943|        18.84|
|HP OfficeJet Pro 8139e Wireless All-in-On